# 2. Model Training

This notebook trains the 5-tier HP prediction models using engineered features.

**Input:** `helper_files/engineered_features.parquet`  
**Output:** `pickled_models/hp_model_cr*.pkl`

## Imports and Configs

In [22]:
import pandas as pd
import numpy as np
import os
import sys
from pathlib import Path
from sklearn.preprocessing import StandardScaler


In [23]:
# Detect execution context and set paths dynamically
sys.path.insert(0, '.')

# Get the current working directory
cwd = Path.cwd()

# Check if we're in the notebooks directory or project root
if cwd.name == 'notebooks':
    # Running from notebooks directory (in Jupyter)
    DATA_DIR = '../data'
    PICKLED_MODELS_DIR = '../pickled_models'
    MONSTER_BUILDER_DIR = '../monster-builder-v2'
    HELPERS_DIR = './helper_files'
    IO_DIR = './notebooks_io'
    IN_NB_DIR = False
else:
    # Running from project root (via run_three_tier_model.py)
    DATA_DIR = './data'
    PICKLED_MODELS_DIR = './pickled_models'
    MONSTER_BUILDER_DIR = './monster-builder-v2'
    HELPERS_DIR = './notebooks/helper_files'
    IO_DIR = './notebooks/notebooks_io'
    IN_NB_DIR = False

print(f"📁 Execution context detected:")
print(f"   Current directory: {cwd}")
print(f"   Data directory: {DATA_DIR}")
print(f"   Models directory: {PICKLED_MODELS_DIR}")

print("Imports successful")

📁 Execution context detected:
   Current directory: /workspaces/matrix_v0
   Data directory: ./data
   Models directory: ./pickled_models
Imports successful


In [24]:
# Add helper_files to path
if IN_NB_DIR is True:
    from helper_files import (
        get_phase3_features,
        train_constrained_model,
        ConstrainedModel,
        calculate_r2,
        calculate_mae,
        save_model,
        summarize_model_performance,
        extract_family,
        PHASE2_FEATURES,
        PHASE2_PENALTIES,
        add_percentile_by_cr,
        log_model_performance,
    )

    print("Imports successful")
else:
    from notebooks.helper_files import (
        get_phase3_features,
        train_constrained_model,
        ConstrainedModel,
        calculate_r2,
        calculate_mae,
        save_model,
        summarize_model_performance,
        extract_family,
        PHASE2_FEATURES,
        PHASE2_PENALTIES,
        add_percentile_by_cr,
        log_model_performance,
    )  

    print("Imports successful")

Imports successful


## Load Engineered Features

In [25]:
# Load engineered features
load_path = IO_DIR + "/engineered_features.parquet"
df = pd.read_parquet(load_path)
print(f"Loaded {len(df)} monsters with {len(df.columns)} features")

Loaded 324 monsters with 130 features


In [26]:
# Get Phase 3 features
phase3_features = get_phase3_features()
print(f"Phase 3 features: {len(phase3_features)}")

Phase 3 features: 38


## Split by CR Tier

In [27]:
# Split by CR tier
df_cr1 = df[df['cr_tier'] == 'cr1'].copy()
df_cr2 = df[df['cr_tier'] == 'cr2'].copy()
df_cr3 = df[df['cr_tier'] == 'cr3'].copy()
df_cr4 = df[df['cr_tier'] == 'cr4'].copy()
df_cr5 = df[df['cr_tier'] == 'cr5'].copy()

print(f"CR < 1:    {len(df_cr1)} monsters")
print(f"CR 1-4:    {len(df_cr2)} monsters")
print(f"CR 5-10:   {len(df_cr3)} monsters")
print(f"CR 11-16:  {len(df_cr4)} monsters")
print(f"CR > 16:   {len(df_cr5)} monsters")

CR < 1:    113 monsters
CR 1-4:    99 monsters
CR 5-10:   65 monsters
CR 11-16:  27 monsters
CR > 16:   20 monsters


## Train/Test Strategy

**Matching original notebook behavior:** Using ALL data for both training and testing.
This measures model fit rather than generalization, which is intentional for this use case.

In [28]:
# Use ALL data for both training and testing (matching original notebook)
# This measures fit rather than generalization, which is intentional

def split_by_tier(df_tier, tier_name):
    """Return same data for both train and test (no split)."""
    print(f"  {tier_name}: Using all {len(df_tier)} samples for both training and testing")
    return df_tier, df_tier

In [29]:
# Split each tier
print("Splitting data:")
train_cr1, test_cr1 = split_by_tier(df_cr1, 'CR < 1')
train_cr2, test_cr2 = split_by_tier(df_cr2, 'CR 1-4')
train_cr3, test_cr3 = split_by_tier(df_cr3, 'CR 5-10')
train_cr4, test_cr4 = split_by_tier(df_cr4, 'CR 11-16')
train_cr5, test_cr5 = split_by_tier(df_cr5, 'CR > 16')

Splitting data:
  CR < 1: Using all 113 samples for both training and testing
  CR 1-4: Using all 99 samples for both training and testing
  CR 5-10: Using all 65 samples for both training and testing
  CR 11-16: Using all 27 samples for both training and testing
  CR > 16: Using all 20 samples for both training and testing


In [30]:
df_cr1.shape

(113, 130)

## Train Models

In [31]:
def train_tier_model(train_df, test_df, tier_name, phase3_features):
    """Train a model for a single CR tier."""
    print(f"\n{'='*60}")
    print(f"Training {tier_name} model...")
    print(f"{'='*60}")
    
    # Prepare features
    X_train = train_df[phase3_features].fillna(0).values
    y_train = train_df['residual_hp'].values
    
    X_test = test_df[phase3_features].fillna(0).values
    y_test = test_df['residual_hp'].values
    
    # Scale features (with_mean=False to preserve zero values)
    scaler = StandardScaler(with_mean=False)
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # Train constrained model
    coefficients, intercept = train_constrained_model(
        X_train_scaled, y_train, phase3_features, scaler
    )
    
    # Create model object
    model = ConstrainedModel(coefficients, intercept)
    
    # Evaluate on test set
    y_pred_residual = model.predict(X_test_scaled)
    
    # Calculate full HP predictions (residual is based on hp_after_resist_immun_penalty)
    y_pred_hp = test_df['hp_after_resist_immun_penalty'].values + y_pred_residual
    y_actual_hp = test_df['actual_hp'].values
    
    # Calculate metrics
    r2 = calculate_r2(y_actual_hp, y_pred_hp)
    mae = calculate_mae(y_actual_hp, y_pred_hp)
    
    print(f"\n{tier_name} Results:")
    print(f"   Training samples: {len(y_train)}")
    print(f"   Test R²:  {r2:.4f}")
    print(f"   Test MAE: {mae:.2f} HP")
    
    return {
        'model': model,
        'scaler': scaler,
        'train_count': len(y_train),
        'test_r2': r2,
        'test_mae': mae,
    }

In [32]:
# Train all models
results = {}

results['cr1'] = train_tier_model(train_cr1, test_cr1, 'CR < 1', phase3_features)
results['cr2'] = train_tier_model(train_cr2, test_cr2, 'CR 1-4', phase3_features)
results['cr3'] = train_tier_model(train_cr3, test_cr3, 'CR 5-10', phase3_features)
results['cr4'] = train_tier_model(train_cr4, test_cr4, 'CR 11-16', phase3_features)
results['cr5'] = train_tier_model(train_cr5, test_cr5, 'CR > 16', phase3_features)


Training CR < 1 model...

CR < 1 Results:
   Training samples: 113
   Test R²:  0.4590
   Test MAE: 4.75 HP

Training CR 1-4 model...

CR 1-4 Results:
   Training samples: 99
   Test R²:  0.5626
   Test MAE: 11.34 HP

Training CR 5-10 model...



CR 5-10 Results:
   Training samples: 65
   Test R²:  0.4592
   Test MAE: 18.30 HP

Training CR 11-16 model...

CR 11-16 Results:
   Training samples: 27
   Test R²:  1.0000
   Test MAE: 0.00 HP

Training CR > 16 model...

CR > 16 Results:
   Training samples: 20
   Test R²:  0.9559
   Test MAE: 15.68 HP


In [33]:
results['cr5'].keys()

dict_keys(['model', 'scaler', 'train_count', 'test_r2', 'test_mae'])

# Final Combinations and Saves

## Save Models

In [34]:
# Ensure output directory exists
# os.makedirs(PICKLED_MODELS_DIR, exist_ok=True)
save_path = PICKLED_MODELS_DIR + "/hp_model_tier.pkl"
# Save each model
for tier in ['cr1', 'cr2', 'cr3', 'cr4', 'cr5']:

    filepath = save_path.replace('tier', tier)
    save_model(
        results[tier]['model'],
        results[tier]['scaler'],
        phase3_features,
        results[tier]['train_count'],
        results[tier]['test_r2'],
        results[tier]['test_mae'],
        filepath
    )
    print(f"Saved {tier} model to {filepath}")

print("\nAll models saved successfully!")

Saved cr1 model to ./pickled_models/hp_model_cr1.pkl
Saved cr2 model to ./pickled_models/hp_model_cr2.pkl
Saved cr3 model to ./pickled_models/hp_model_cr3.pkl
Saved cr4 model to ./pickled_models/hp_model_cr4.pkl
Saved cr5 model to ./pickled_models/hp_model_cr5.pkl

All models saved successfully!


## Generate Predictions

In [35]:
# Build models and scalers dicts from training results
models = {tier: results[tier]['model'] for tier in ['cr1', 'cr2', 'cr3', 'cr4', 'cr5']}
scalers = {tier: results[tier]['scaler'] for tier in ['cr1', 'cr2', 'cr3', 'cr4', 'cr5']}

def get_prediction_for_creature(row):
    """Get the final HP prediction for a creature based on its CR tier."""
    tier = row['cr_tier']
    
    # Get features
    X = row[phase3_features].to_frame().T.fillna(0).infer_objects(copy=False).values.reshape(1, -1)
    
    # Scale and predict
    X_scaled = scalers[tier].transform(X)
    residual_pred = models[tier].predict(X_scaled)[0]
    
    # Residual is based on hp_after_resist_immun_penalty
    return row['hp_after_resist_immun_penalty'] + residual_pred

df['predicted_hp'] = df.apply(get_prediction_for_creature, axis=1)
df['hp_delta'] = df['predicted_hp'] - df['actual_hp']
df['hp_delta_pct'] = (df['hp_delta'] / df['actual_hp']) * 100

df = add_percentile_by_cr(df)

print("Predictions generated")

/tmp/ipykernel_12396/1044953982.py:10: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X = row[phase3_features].to_frame().T.fillna(0).infer_objects(copy=False).values.reshape(1, -1)
/tmp/ipykernel_12396/1044953982.py:10: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X = row[phase3_features].to_frame().T.fillna(0).infer_objects(copy=False).values.reshape(1, -1)
/tmp/ipykernel_12396/1044953982.py:10: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=Fal

Predictions generated


In [36]:
# Summary statistics
print("Overall Prediction Summary:")
print(f"   Mean HP Error: {df['hp_delta'].mean():.1f} HP")
print(f"   Mean Absolute Error: {df['hp_delta'].abs().mean():.1f} HP")
print(f"   Mean % Error: {df['hp_delta_pct'].mean():.1f}%")
print(f"   Mean Absolute % Error: {df['hp_delta_pct'].abs().mean():.1f}%")

Overall Prediction Summary:
   Mean HP Error: 0.2 HP
   Mean Absolute Error: 9.8 HP
   Mean % Error: 1.0%
   Mean Absolute % Error: 41.1%


In [37]:
# Summary by CR tier
tier_labels = {
    'cr1': 'CR < 1',
    'cr2': 'CR 1-4',
    'cr3': 'CR 5-10',
    'cr4': 'CR 11-16',
    'cr5': 'CR > 16',
}

print("\nBy CR Tier:")
for tier in ['cr1', 'cr2', 'cr3', 'cr4', 'cr5']:
    tier_df = df[df['cr_tier'] == tier]
    mae = tier_df['hp_delta'].abs().mean()
    mape = tier_df['hp_delta_pct'].abs().mean()
    print(f"   {tier_labels[tier]:12s}: MAE={mae:6.1f} HP, MAPE={mape:5.1f}%")


By CR Tier:
   CR < 1      : MAE=   4.7 HP, MAPE= 83.1%
   CR 1-4      : MAE=  11.3 HP, MAPE= 27.9%
   CR 5-10     : MAE=  18.3 HP, MAPE= 17.0%
   CR 11-16    : MAE=   0.0 HP, MAPE=  0.0%
   CR > 16     : MAE=  15.7 HP, MAPE=  4.1%


## Feature Contributions

In [38]:
from notebooks.helper_files import PHASE2_FEATURES

def calculate_feature_contributions(row):
    """Calculate HP contribution of each feature for a creature."""
    tier = row['cr_tier']
    penalties = PHASE2_PENALTIES[tier]
    
    contributions = {
        'Name': row['Name'],
        'CR': row['cr_numeric'],
        'actual_hp': row['actual_hp'],
        'predicted_hp': row['predicted_hp'],
        'hp_error': row['hp_delta'],
        'hp_error_pct': row['hp_delta_pct'],
        'hp_baseline': row['hp_baseline'],
        'hp_after_phase2': row['hp_after_phase2'],
        'hp_after_resist_immun_penalty': row['hp_after_resist_immun_penalty'],
        'resist_immun_resistance_penalty': row.get('resistance_penalty', 0),
        'resist_immun_immunity_penalty': row.get('immunity_penalty', 0),
        'resist_immun_total_penalty': row.get('total_defensive_penalty', 0),
    }
    
    # Add all Phase 2 feature RAW VALUES
    for feature in PHASE2_FEATURES:
        contributions[feature] = row.get(feature, 0)
    
    # Phase 2 contributions (calculated HP impact)
    contributions['phase2_ac_contribution'] = row['ac_deviation'] * penalties.get('ac_deviation', 0)
    contributions['phase2_attack_contribution'] = row['attack_deviation'] * penalties.get('attack_deviation', 0)
    contributions['phase2_dpr_contribution'] = row['dpr_deviation'] * penalties.get('dpr_deviation', 0)
    contributions['phase2_save_dc_contribution'] = row['save_dc_deviation'] * penalties.get('save_dc_deviation', 0)
    contributions['phase2_flying_contribution'] = row['has_flying'] * penalties.get('has_flying', 0)
    contributions['phase2_advantage_contribution'] = row.get('has_advantage_condition', 0) * penalties.get('has_advantage_condition', 0)
    contributions['phase2_disadvantage_contribution'] = row.get('has_disadvantage_condition', 0) * penalties.get('has_disadvantage_condition', 0)
    contributions['phase2_attackers_advantage_contribution'] = row.get('has_attackers_advantage', 0) * penalties.get('has_attackers_advantage', 0)
    contributions['phase2_prone_contribution'] = row.get('inflicts_prone', 0) * penalties.get('inflicts_prone', 0)
    
    contributions['phase2_total_contribution'] = sum([
        contributions['phase2_ac_contribution'],
        contributions['phase2_attack_contribution'],
        contributions['phase2_dpr_contribution'],
        contributions['phase2_save_dc_contribution'],
        contributions['phase2_flying_contribution'],
        contributions['phase2_advantage_contribution'],
        contributions['phase2_disadvantage_contribution'],
        contributions['phase2_attackers_advantage_contribution'],
        contributions['phase2_prone_contribution'],
    ])
    
    # Add all Phase 3 feature RAW VALUES
    for feature in phase3_features:
        contributions[feature] = row.get(feature, 0)
    
    # Phase 3 contributions (calculated HP impact)
    X = np.array([[row.get(f, 0) for f in phase3_features]])
    X = np.nan_to_num(X, 0)
    X_scaled = scalers[tier].transform(X)[0]
    coefs = models[tier].coef_
    
    phase3_total = 0
    for i, feature in enumerate(phase3_features):
        contrib = X_scaled[i] * coefs[i]
        contributions[f'phase3_{feature}'] = contrib
        phase3_total += contrib
    
    contributions['phase3_intercept'] = models[tier].intercept_
    contributions['phase3_total_contribution'] = phase3_total + contributions['phase3_intercept']
    
    return pd.Series(contributions)

# Calculate contributions for all creatures
contributions_df = df.apply(calculate_feature_contributions, axis=1)
print(f"Calculated contributions for {len(contributions_df)} creatures")
print(f"Columns include: {len([c for c in contributions_df.columns if c in PHASE2_FEATURES])} Phase 2 features, {len([c for c in contributions_df.columns if c in phase3_features])} Phase 3 features")

Calculated contributions for 324 creatures
Columns include: 9 Phase 2 features, 38 Phase 3 features


## Export Completed Data

In [39]:
# Export engineered features
export_columns = [
    'Name', 'Type', 'Size', 'Challenge_Rating', 'cr_numeric', 'cr_tier',
    'HP',  
    'hp_baseline','actual_hp', 'predicted_hp', 'hp_delta', 'hp_delta_pct', 'hp_delta_pct_percentile',
    'AC', 'ac_value', 'ac_baseline','ac_deviation',
    
    'dc_baseline','highest_save_dc','save_dc_deviation',
    'attack_baseline', 'highest_attack_bonus', 'attack_deviation',

    'dpr_baseline',  'estimated_dpr', 'legendary_dpr', 'total_dpr', 'dpr_deviation',
       
    'hp_after_phase2', 'hp_after_resist_immun_penalty', 'residual_hp',
]
for col in PHASE2_FEATURES:
    if col not in export_columns: 
       export_columns.append(col)
for col in phase3_features:
    if col not in export_columns: 
       export_columns.append(col)
       
# Add all existing columns that match
export_cols = [c for c in export_columns if c in df.columns]

export_df = df[export_cols].sort_values('cr_numeric')

export_path = DATA_DIR + '/engineered_features.csv'
export_df.to_csv(export_path, index=False)
print(f"Exported {len(export_df)} monsters with {len(export_cols)} features to data/engineered_features.csv")

Exported 324 monsters with 74 features to data/engineered_features.csv


In [40]:
# Export contributions
export_path = DATA_DIR + '/feature_contributions.csv'
contributions_df.to_csv(export_path, index=False)
print(f"Exported contributions to data/feature_contributions.csv")

Exported contributions to data/feature_contributions.csv


# Record Performance

In [41]:
# Prompt user for change description
print("=" * 60)
print("📝 MODEL CHANGE LOG")
print("=" * 60)
try:
    CHANGE_SUMMARY = input("Enter a brief description of changes made in this run ('skip' to skip): ")
except:
    CHANGE_SUMMARY = "(automated run - no description provided)"

# Log the performance
log_model_performance(CHANGE_SUMMARY, results, IN_NB_DIR)

📝 MODEL CHANGE LOG
   R² scores (full HP prediction):
      CR<1:    0.4590
      CR1-4:   0.5626
      CR5-10:  0.4592
      CR11-16: 1.0000
      CR>16:   0.9559
⚠️  Skipping model performance logging as per user request.
